# 00 · Validación de coordenadas, cobertura e índice por tienda

**Objetivo:** Mostrar a directores y dueños si la competencia cercana cambia el precio de una tienda, como caso de competencia local para decisiones de expansión y de precio por tienda.

**Pregunta:** Una tienda de Walmart de México o Soriana con un competidor de otra cadena a menos de 2 km, ¿tiene precios distintos a una sin competidor cerca en la misma ciudad?

Este notebook responde, antes de cualquier estimación: ¿cuántas tiendas tienen coordenadas confiables y hay
suficientes tiendas **con y sin** competidor cercano **en la misma ciudad**? Si no alcanzara, el análisis se detendría
aquí. Cierra con la tabla de decisiones metodológicas (plan aprobado en `_specs/plan_metodologico_6_12.md`).

**«Sin competidor cerca» significa «sin competidor monitoreado por PROFECO cerca».** Puede haber una tienda que PROFECO
no visita a 500 metros. Y es correlación, no causalidad: las cadenas no abren tiendas al azar.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import competencia as K
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
REPS = 500
raw = entorno.load_raw()
manifest = entorno.write_manifest(raw)
pts, st = K.load()
allst = entorno.tabla("tiendas")
print(f"Tiendas en el archivo: {len(allst):,} · con coordenada confiable: {allst.coord_ok.sum():,}")
display(allst.coord_motivo.replace("", "confiable").value_counts().to_frame("tiendas"))
print(f"Supermercados: {len(st)} · con coordenada confiable: {st.coord_ok.sum()}")

Tiendas en el archivo: 1,857 · con coordenada confiable: 1,692


,tiendas
coord_motivo,
confiable,1692
compartida con otra tienda,158
sin coordenadas,3
fuera de México,2
a más de 30 km de su municipio,2


Supermercados: 319 · con coordenada confiable: 315


## 1. Coordenadas

**Decisión:** usar solo tiendas con `coord_ok` (tabla común `tiendas`). La bandera descarta, sin borrar del archivo, las
tiendas sin coordenada, fuera de México, con coordenada que cambia entre visitas, compartida con otra tienda (señal de
que se geocodificó al centro de la ciudad) o a más de 30 km de su municipio. Alternativa descartada: corregir
coordenadas a mano, porque sería inventar ubicaciones.

In [2]:
idx = K.store_index(pts, st)
foc = K.focal_stores(st, idx)
print(f"Tiendas focales (Walmart de México, Soriana y Chedraui) con coordenada confiable e índice: {len(foc)}")
cov = K.coverage(foc)
display(cov)

Tiendas focales (Walmart de México, Soriana y Chedraui) con coordenada confiable e índice: 247


,radio_km,grupo,tiendas,con_competidor,sin_competidor,sin_en_ciudades_con_ambos,ciudades_con_ambos
0,1,Walmart de México,130,46,84,46,30
1,1,Soriana,87,35,52,27,28
2,1,Chedraui,30,9,21,8,15
3,1,Total,247,90,157,81,30
4,2,Walmart de México,130,92,38,33,29
5,2,Soriana,87,70,17,16,29
6,2,Chedraui,30,27,3,2,13
7,2,Total,247,189,58,51,29
8,3,Walmart de México,130,114,16,14,15
9,3,Soriana,87,80,7,7,15


## 2. ¿Alcanzan los datos?

Para comparar dentro de la misma ciudad se necesitan ciudades con tiendas de los dos tipos.
- **A 2 km** hay 29 ciudades así. Walmart de México y Soriana tienen tiendas sin competidor en esas ciudades.
- **Chedraui** casi no tiene tiendas sin competidor (2 en esas ciudades): **no se estima su efecto**, como preveía el
  plan. Sus tiendas entran a la regresión, pero solo para ayudar a estimar el nivel de cada ciudad.
- **A 3 km** casi no quedan tiendas sin competidor: ese radio es solo sensibilidad, con poca potencia.

**Los datos alcanzan para Walmart de México y Soriana, con intervalos que pueden ser amplios.**

## 3. Índice de precio de cada tienda

**Decisión:** índice = promedio, sobre todos los SKUs de la tienda, de log(precio de la tienda / precio promedio del
mismo SKU en las otras tiendas de supermercado del municipio). Así se compara la tienda con su ciudad usando sus
propios productos (la canasta que comparte con su ciudad). Alternativa descartada: índice contra la mediana nacional,
porque mezclaría ciudad y competencia.

In [3]:
display(foc.groupby("grupo")[["skus", "indice"]].describe().round(3))
foc.to_csv(PROC / "tiendas_focales.csv", index=False, encoding="utf-8-sig")
cov.to_csv(PROC / "cobertura_por_radio.csv", index=False, encoding="utf-8-sig")
params = {"semilla": K.SEMILLA, "bootstrap_replicas": REPS, "canal": K.CANAL, "focales": K.FOCALES,
          "evaluables": K.EVALUABLES, "radios_km": K.RADIOS, "radio_principal_km": K.RADIO, "min_sin": K.MIN_SIN}
_ = (META / "parametros.json").write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
F = "00_validacion_y_datos"
reg("tiendas_total", len(allst), fuente=F); reg("coord_ok_total", allst.coord_ok.sum(), fuente=F)
reg("super_total", len(st), fuente=F); reg("super_coord_ok", st.coord_ok.sum(), fuente=F)
reg("focales", len(foc), fuente=F); reg("skus_mediana", foc.skus.median(), fuente=F)
for _, r_ in cov.iterrows():
    k = f"{K.TIPOS.get(r_.grupo, 'total')}_{r_.radio_km}km"
    reg(f"con_{k}", r_.con_competidor, fuente=F); reg(f"sin_{k}", r_.sin_competidor, fuente=F)
    reg(f"sinamb_{k}", r_.sin_en_ciudades_con_ambos, fuente=F); reg(f"ciud_{k}", r_.ciudades_con_ambos, fuente=F)

skus                                                           indice                                                 
                   count     mean      std    min     25%     50%      75%     max  count   mean    std    min    25%    50%    75%    max
grupo                                                                                                                                     
Chedraui            30.0  973.233  162.194  456.0  875.25   984.5  1060.00  1373.0   30.0 -0.049  0.015 -0.083 -0.056 -0.052 -0.045 -0.001
Soriana             87.0  976.690  333.971   57.0  809.00  1061.0  1181.50  1627.0   87.0  0.031  0.023 -0.067  0.019  0.032  0.044  0.083
Walmart de México  130.0  842.254  341.655   39.0  739.50   928.0  1066.75  1405.0  130.0 -0.015  0.024 -0.075 -0.032 -0.015  0.001  0.061

## 4. Decisiones metodológicas

In [4]:
display(Markdown(f'''
| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Ubicación | solo `coord_ok` ({st.coord_ok.sum()} de {len(st)} supermercados) | coordenadas compartidas o lejanas no ubican la tienda | corregir coordenadas a mano |
| 2 | Competidor | supermercado de otro grupo monitoreado por PROFECO, a 2 km o menos (sensibilidad 1 y 3 km) | lo que el archivo permite observar | contar tiendas no monitoreadas (no hay datos) |
| 3 | Índice por tienda | promedio de log(precio / promedio del mismo SKU en las otras tiendas del municipio) | mismo producto, misma ciudad | índice contra la mediana nacional |
| 4 | Comparación | regresión del índice con efectos fijos de municipio y cadena (formato) | compara tiendas de la misma cadena en la misma ciudad | comparar tiendas de ciudades distintas |
| 5 | Grupos | Walmart de México y Soriana; Chedraui no evaluable (casi sin tiendas sin competidor) | sin tiendas de los dos tipos no hay comparación | forzar una cifra para Chedraui |
| 6 | Incertidumbre | IC90 por bootstrap de municipios, 500 réplicas, semilla 2026 | las tiendas de una ciudad no son independientes | errores por tienda |
| 7 | Variantes | tipo de competidor, radio de 1, 2 y 3 km, distancia continua | ¿quién compite y a qué distancia importa? | — |
| — | Alcance | correlación, no causalidad; «sin competidor» = sin competidor monitoreado | las tiendas no se ubican al azar | afirmar que la competencia baja precios |
'''))


| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Ubicación | solo `coord_ok` (315 de 319 supermercados) | coordenadas compartidas o lejanas no ubican la tienda | corregir coordenadas a mano |
| 2 | Competidor | supermercado de otro grupo monitoreado por PROFECO, a 2 km o menos (sensibilidad 1 y 3 km) | lo que el archivo permite observar | contar tiendas no monitoreadas (no hay datos) |
| 3 | Índice por tienda | promedio de log(precio / promedio del mismo SKU en las otras tiendas del municipio) | mismo producto, misma ciudad | índice contra la mediana nacional |
| 4 | Comparación | regresión del índice con efectos fijos de municipio y cadena (formato) | compara tiendas de la misma cadena en la misma ciudad | comparar tiendas de ciudades distintas |
| 5 | Grupos | Walmart de México y Soriana; Chedraui no evaluable (casi sin tiendas sin competidor) | sin tiendas de los dos tipos no hay comparación | forzar una cifra para Chedraui |
| 6 | Incertidumbre | IC90 por bootstrap de municipios, 500 réplicas, semilla 2026 | las tiendas de una ciudad no son independientes | errores por tienda |
| 7 | Variantes | tipo de competidor, radio de 1, 2 y 3 km, distancia continua | ¿quién compite y a qué distancia importa? | — |
| — | Alcance | correlación, no causalidad; «sin competidor» = sin competidor monitoreado | las tiendas no se ubican al azar | afirmar que la competencia baja precios |


In [5]:
c2 = cov[(cov.radio_km == 2)].set_index("grupo")
display(Markdown(
    f"**Conclusión.** Hay {len(foc)} tiendas focales con coordenada confiable e índice. A 2 km, {c2.loc['Total', 'con_competidor']} tienen competidor "
    f"y {c2.loc['Total', 'sin_competidor']} no; {c2.loc['Total', 'ciudades_con_ambos']} ciudades tienen tiendas de los dos tipos. Alcanza para "
    f"Walmart de México ({c2.loc['Walmart de México', 'sin_en_ciudades_con_ambos']} tiendas sin competidor en esas ciudades) y Soriana "
    f"({c2.loc['Soriana', 'sin_en_ciudades_con_ambos']}); no para Chedraui ({c2.loc['Chedraui', 'sin_en_ciudades_con_ambos']})."))

**Conclusión.** Hay 247 tiendas focales con coordenada confiable e índice. A 2 km, 189 tienen competidor y 58 no; 29 ciudades tienen tiendas de los dos tipos. Alcanza para Walmart de México (33 tiendas sin competidor en esas ciudades) y Soriana (16); no para Chedraui (2).